<a href="https://colab.research.google.com/github/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/blob/main/my%20training%20code/step5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install setfit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [5]:
import sys
import transformers.training_args
transformers.training_args.default_logdir = lambda: "tmp_trainer"

In [6]:
import pandas as pd
import numpy as np
import torch
import time
import urllib.request
import yaml
import re
from setfit import SetFitModel

print("🚀 啟動階段五：模型大規模盲測與終極 OOD 防禦系統評估")

# ==========================================
# 0. 雲端化防禦設定載入 (確保與 Step 4 100% 一致)
# ==========================================
print("\n正在從遠端 GitHub 載入最新防禦設定與豁免白名單...")
YAML_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/keywords.yaml"
IGNORE_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/ignore.txt"

# A. 動態地名脫敏器
def get_sanitizer(yaml_url):
    response = urllib.request.urlopen(yaml_url)
    config = yaml.safe_load(response)
    loc_config = config.get('locations', {})
    all_locations = []
    for cat in loc_config:
        all_locations.extend(loc_config[cat])
    if not all_locations: return None
    return re.compile(f"({'|'.join(all_locations)})")

city_pattern = get_sanitizer(YAML_URL)

def preprocess_texts(texts, replacement="[LOC]"):
    if not city_pattern: return texts
    if isinstance(texts, str): return city_pattern.sub(replacement, texts)
    return [city_pattern.sub(replacement, str(t)) if pd.notna(t) else "" for t in texts]

# B. 載入豁免清單
ignore_response = urllib.request.urlopen(IGNORE_URL)
ignore_keywords = [line.decode('utf-8').strip() for line in ignore_response if line.strip() and not line.decode('utf-8').startswith('#')]


# ==========================================
# 1. 載入模型與大規模盲測資料
# ==========================================
print("\n正在載入你剛剛微調成功的專屬 SetFit 模型...")
model = SetFitModel.from_pretrained("Alyssalai/my_intent_classifier_setfit")

print("正在從遠端載入大量候選池資料進行實戰盲測...")
# 💡 修正點：改用雲端網址直接讀取，防止本地 FileNotFoundError
CANDIDATE_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/candidate_pool_sanitized.csv"
df_test = pd.read_csv(CANDIDATE_URL)
df_test = df_test.dropna(subset=['name']).drop_duplicates(subset=['name']).reset_index(drop=True)


# ==========================================
# 2. 執行全量盲測推論
# ==========================================
# 實戰推論前，文字必須先經過地名脫敏清洗，才符合大模型的認知！
raw_texts = df_test['name'].astype(str).tolist()
sanitized_texts = preprocess_texts(raw_texts)

print(f"🔥 開始對 {len(sanitized_texts)} 筆未知社團資料進行批次推論...")
start_time = time.time()
preds = model.predict(sanitized_texts)
print(f"✅ 大規模推論完成！總耗時: {time.time() - start_time:.2f} 秒")

df_test['setfit_predict'] = preds


# ==========================================
# 3. 盲測對比分析 (Snorkel vs SetFit)
# ==========================================
# 如果這份 CSV 裡面有之前 Snorkel 留下的預測欄位，我們就拿來比對
if 'snorkel_label' in df_test.columns:
    df_rescued = df_test[df_test['snorkel_label'].isna() | (df_test['snorkel_label'] == '放棄投票') | (df_test['snorkel_label'] == '棄權未分類')]
    df_diff = df_test[
        (df_test['snorkel_label'].notna()) &
        (df_test['snorkel_label'] != '放棄投票') &
        (df_test['snorkel_label'] != '棄權未分類') &
        (df_test['snorkel_label'] != df_test['setfit_predict'])
    ]
else:
    df_rescued = pd.DataFrame()
    df_diff = pd.DataFrame()

print("\n" + "="*50)
print("🎯 隨機抽樣盲測預測結果展示")
print("="*50)
for _, row in df_test.sample(min(10, len(df_test)), random_state=0).iterrows():
    print(f"[{row['setfit_predict']}] <- {row['name']}")

if not df_rescued.empty:
    print("\n" + "="*50)
    print(f"💡 模型顯靈：從 Snorkel 棄權的資料中，完美解纏並分類了 {len(df_rescued)} 筆！(隨機抽樣 5 筆)")
    print("="*50)
    for _, row in df_rescued.sample(min(5, len(df_rescued))).iterrows():
        print(f"文本: {row['name']} -> 預判: [{row['setfit_predict']}]")

if not df_diff.empty:
    print("\n" + "="*50)
    print(f"⚔️ 判決逆轉：SetFit 泛化能力展現！推翻 Snorkel 硬性規則共有 {len(df_diff)} 筆！(隨機抽樣 5 筆)")
    print("="*50)
    for _, row in df_diff.sample(min(5, len(df_diff))).iterrows():
        print(f"文本: {row['name']}")
        print(f"原規則標籤: [{row['snorkel_label']}] -> 新大模型標籤: [{row['setfit_predict']}]\n")

# 儲存盲測結果
output_file = 'candidate_pool_with_setfit_predictions.csv'
df_test.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"💾 完整盲測結果已儲存至本地檔案: {output_file}")


# ==========================================
# 4. 終極防禦系統：整合 OOD 與 Ignore 機制
# ==========================================
def smart_classify(raw_texts, threshold=0.65):
    # 先做地名脫敏
    texts = preprocess_texts(raw_texts)

    # 取得大模型的信心度預測機率分佈
    probs = model.predict_proba(texts)
    preds = model.predict(texts)

    results = []
    for i, raw_text in enumerate(raw_texts):
        clean_text = texts[i]
        max_prob = probs[i].max().item()
        label = preds[i]

        # 🛡️ 關鍵補強：第一道防線 - 硬性白名單攔截
        if any(ignore_word in raw_text for ignore_word in ignore_keywords):
            final_label = "正常困難負樣本"
            max_prob = 1.0000  # 白名單命中，給予最高信心度
        # 🤖 第二道防線 - 信心度判定邏輯 (OOD 攔截)
        elif max_prob < threshold:
            final_label = "異常/未知 (OOD)"
        else:
            final_label = label

        results.append({
            "原始文本": raw_text,
            "脫敏文本": clean_text,
            "最終預測": final_label,
            "信心度": round(max_prob, 4),
            "模型原生猜測": label
        })
    return pd.DataFrame(results)

# 進行特定邊界案例測試
print("\n" + "="*50)
print("📊 最終防禦系統展示測試（含 OOD 與 豁免防線）")
print("="*50)
test_data = [
    "蔣公（Chiang Kai-shek）民族英雄 抗日領袖 時代巨人 自由燈塔", # OOD 測試
    "專業快速撥款，免保人免留車，全台皆可辦", # 借貸融資
    "台北萬豪酒店", # 觸發白名單與地名脫敏
    "加入自救會，律師教你如何拿回被騙的錢", # 臨界或 OOD 測試
    "今日天氣晴朗，適合去陽明山踏青", # OOD 測試
    "台南信貸", # 地名脫敏測試
    "八大" # 邊界詞彙
]

df_final = smart_classify(test_data)
display(df_final)

🚀 啟動階段五：模型大規模盲測與終極 OOD 防禦系統評估

正在從遠端 GitHub 載入最新防禦設定與豁免白名單...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



正在載入你剛剛微調成功的專屬 SetFit 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ 大規模推論完成！總耗時: 96.82 秒

🎯 隨機抽樣盲測預測結果展示
[黃色與特種行業] <- 不動產經紀人莊意輝買屋賣屋
[博弈] <- 運彩情報分析站
[黃色與特種行業] <- 皇家國際商務酒店【Royal】
[黃色與特種行業] <- 淚語公關部
[黃色與特種行業] <- 杜拜演藝經紀
[博弈] <- Super8線上娛樂城
[借貸融資] <- 兆豐國際理財貸款
[黃色與特種行業] <- 大高雄房地-永慶 盧麗玲 專業經紀人 0933289089
[借貸融資] <- 關於買賣車或貸款融資方面的困擾找我就對了⋯⋯
[黃色與特種行業] <- 台大酒店小姐手札
💾 完整盲測結果已儲存至本地檔案: candidate_pool_with_setfit_predictions.csv

📊 最終防禦系統展示測試（含 OOD 與 豁免防線）


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,原始文本,脫敏文本,最終預測,信心度,模型原生猜測
0,蔣公（Chiang Kai-shek）民族英雄 抗日領袖 時代巨人 自由燈塔,蔣公（Chiang Kai-shek）民族英雄 抗日領袖 時代巨人 自由燈塔,詐騙高風險招募,0.9957,詐騙高風險招募
1,專業快速撥款，免保人免留車，全台皆可辦,專業快速撥款，免保人免留車，全台皆可辦,借貸融資,0.9985,借貸融資
2,台北萬豪酒店,[LOC]萬豪酒店,黃色與特種行業,0.9990,黃色與特種行業
3,加入自救會，律師教你如何拿回被騙的錢,加入自救會，律師教你如何拿回被騙的錢,正常困難負樣本,1.0000,借貸融資
4,今日天氣晴朗，適合去陽明山踏青,今日天氣晴朗，適合去陽明山踏青,正常困難負樣本,0.9971,正常困難負樣本
5,台南信貸,[LOC]信貸,借貸融資,0.9989,借貸融資
6,八大,八大,正常困難負樣本,0.9983,正常困難負樣本


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
import pandas as pd
import numpy as np
import torch
import time
import urllib.request
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from setfit import SetFitModel

print("👥 啟動階段：五人模型決策小組 (在地學術基線強化防禦版)")

# ==========================================
# 1. 載入實戰測試候選池
# ==========================================
print("\n📋 正在載入大規模盲測候選池資料...")
CANDIDATE_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/candidate_pool_sanitized.csv"
df_vote = pd.read_csv(CANDIDATE_URL)
df_vote = df_vote.dropna(subset=['name']).drop_duplicates(subset=['name']).reset_index(drop=True)

# 為了加快展示與確保穩健，我們直接拿前 200 筆未知社團來做投票矩陣評估
df_vote = df_vote.head(200).copy()
test_texts = df_vote['name'].astype(str).tolist()

# ==========================================
# 2. 喚醒五位評委 (含在地動態訓練傳統 ML 基線)
# ==========================================
print("\n🔮 正在同步喚醒 5 位評委...")

# 評委 1: 你的雲端 SetFit 模型
HF_USERNAME = "Alyssalai"
repo_id = f"{HF_USERNAME}/my_intent_classifier_setfit"
print(f"-> 評委 1 (SetFit 雲端強化版) 載入中: {repo_id}")
model_setfit = SetFitModel.from_pretrained(repo_id)

# 🛡️ 評委 2 智慧防禦線：動態檢查是否包含 Snorkel 標籤
if 'snorkel_label' in df_vote.columns:
    print("-> 評委 2 (Snorkel 規則派) 已由資料集欄位載入就位")
    df_vote['vote_2_snorkel'] = df_vote['snorkel_label'].fillna("未知").replace("放棄投票", "未知").replace("棄權未分類", "未知")
else:
    print("💡 提示：盲測池中無舊標籤，評委 2 (Snorkel 規則派) 本輪自動選擇「棄權/未知」")
    df_vote['vote_2_snorkel'] = "未知"

# --- 在地動態訓練線：下載訓練集來餵養評委 3, 4, 5 ---
print("-> 正在下載遠端訓練集以初始化在地 ML 評委...")
GOLD_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/gold_candidates_review_with_ignore.csv"
SNORKEL_TRAIN_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/snorkel_labeled_training_data.csv"

df_g = pd.read_csv(GOLD_URL).rename(columns={'proposed_category': 'label_txt'})
df_g['label_txt'] = df_g['label_txt'].map({'loan': '借貸融資', 'porn': '黃色與特種行業', 'gambling': '博弈', 'scam_recruitment': '詐騙高風險招募'})
df_s = pd.read_csv(SNORKEL_TRAIN_URL).rename(columns={'name': 'text', 'snorkel_label': 'label_txt'})

df_train_ml = pd.concat([df_g[['text', 'label_txt']], df_s[['text', 'label_txt']]], ignore_index=True).dropna()
df_train_ml = df_train_ml[df_train_ml['label_txt'] != '毒品']

# 建立 TF-IDF 向量化特徵空間
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(df_train_ml['text'].astype(str))
y_train = df_train_ml['label_txt'].values
X_test = vectorizer.transform(test_texts)

# 評委 3: 邏輯回歸分類器
print("-> 評委 3 (TF-IDF + 邏輯回歸) 在地訓練中...")
clf_lr = LogisticRegression(max_iter=200, random_state=42)
clf_lr.fit(X_train, y_train)

# 評委 4: 多項式單純貝氏分類器
print("-> 評委 4 (TF-IDF + 多項式貝氏) 在地訓練中...")
clf_nb = MultinomialNB()
clf_nb.fit(X_train, y_train)

# 評委 5: 隨機森林分類器
print("-> 評委 5 (TF-IDF + 隨機森林) 在地訓練中...")
clf_rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
clf_rf.fit(X_train, y_train)


# ==========================================
# 3. 評委團正式秘密投票
# ==========================================
print("\n🔥 評委團開始對 200 筆未知社團進行秘密投票...")

df_vote['vote_1_setfit'] = model_setfit.predict(test_texts)
df_vote['vote_3_lr'] = clf_lr.predict(X_test)
df_vote['vote_4_nb'] = clf_nb.predict(X_test)
df_vote['vote_5_rf'] = clf_rf.predict(X_test)


# ==========================================
# 4. 智慧多數決（Majority Voting）共識計算
# ==========================================
def calculate_majority(row):
    votes = [
        row['vote_1_setfit'],
        row['vote_2_snorkel'],
        row['vote_3_lr'],
        row['vote_4_nb'],
        row['vote_5_rf']
    ]
    # 智慧過濾：自動剔除未知與棄權選票
    valid_votes = [v for v in votes if v not in ["未知", "放棄投票", "棄權未分類", None]]

    if not valid_votes:
        return "無法判定", 0.0

    vote_counts = Counter(valid_votes)
    top_label, top_count = vote_counts.most_common(1)[0]
    consensus_rate = top_count / 5.0
    return top_label, consensus_rate

print("📊 正在橫向計算多數決共識矩陣...")
voting_results = df_vote.apply(calculate_majority, axis=1)
df_vote['最終集成標籤'] = [r[0] for r in voting_results]
df_vote['評委共識度'] = [r[1] for r in voting_results]


# ==========================================
# 5. 展示精彩的決策矩陣與學術驗證
# ==========================================
print("\n" + "="*80)
print("🎯 最終五人評委團投票矩陣結果 (前 15 筆隨機抽樣展示)")
print("="*80)

columns_to_show = [
    'name', 'vote_1_setfit', 'vote_2_snorkel',
    'vote_3_lr', 'vote_4_nb', '最終集成標籤', '評委共識度'
]

df_show = df_vote[columns_to_show].sample(15, random_state=42)
df_show.columns = ['社團名稱', 'SetFit(深度語義)', 'Snorkel(規則狀態)', 'LogReg(線性基線)', 'NaiveBayes(機率基線)', '多數決整合解', '評委共識度']
display(df_show)

print("\n📈 【學術統計摘要】")
# 🚨 修正點：改用標準 pandas .sum() 計算，避免舊版語法報錯
high_consensus_pct = (df_vote['評委共識度'] >= 0.6).sum() / len(df_vote) * 100
print(f"1. 在 200 筆測試盲池中，多數模型達成過半數共識（共識度 >= 60%）的比例為: {high_consensus_pct:.2f}%")
print(f"2. 最終多數決整合後，未知池的恶章意圖分佈如下：")
print(df_vote['最終集成標籤'].value_counts())

👥 啟動階段：五人模型決策小組 (在地學術基線強化防禦版)

📋 正在載入大規模盲測候選池資料...

🔮 正在同步喚醒 5 位評委...
-> 評委 1 (SetFit 雲端強化版) 載入中: Alyssalai/my_intent_classifier_setfit


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

💡 提示：盲測池中無舊標籤，評委 2 (Snorkel 規則派) 本輪自動選擇「棄權/未知」
-> 正在下載遠端訓練集以初始化在地 ML 評委...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

📊 正在橫向計算多數決共識矩陣...

🎯 最終五人評委團投票矩陣結果 (前 15 筆隨機抽樣展示)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,社團名稱,SetFit(深度語義),Snorkel(規則狀態),LogReg(線性基線),NaiveBayes(機率基線),多數決整合解,評委共識度
95,長榮鳳凰酒店(礁溪),黃色與特種行業,未知,黃色與特種行業,黃色與特種行業,黃色與特種行業,0.6
15,台灣運彩,博弈,未知,借貸融資,借貸融資,借貸融資,0.6
30,"桃園大溪笠復威斯汀度假酒店 The Westin Tashee Resort, Taoyuan",正常困難負樣本,未知,借貸融資,借貸融資,借貸融資,0.6
158,薆悅酒店 台北-Inhouse Hotel Taipei,黃色與特種行業,未知,黃色與特種行業,借貸融資,黃色與特種行業,0.4
128,運彩紅不讓,博弈,未知,借貸融資,借貸融資,借貸融資,0.6
115,吻鑽時尚酒店,黃色與特種行業,未知,借貸融資,借貸融資,借貸融資,0.6
69,八大男女娛樂交流區,黃色與特種行業,未知,借貸融資,借貸融資,借貸融資,0.6
170,高雄當舖安心推薦-暘銘當舖/汽車借款/機車借款/免留車/黃金鑽石借款,借貸融資,未知,借貸融資,借貸融資,借貸融資,0.8
174,Mr. Beard Car鬍子先生/汽車顧問/高價估車/貸款審核/代客尋車/二手中古汽車銷售,借貸融資,未知,借貸融資,借貸融資,借貸融資,0.8
45,確定你已愛上了他的八大種……,正常困難負樣本,未知,借貸融資,借貸融資,借貸融資,0.6



📈 【學術統計摘要】
1. 在 200 筆測試盲池中，多數模型達成過半數共識（共識度 >= 60%）的比例為: 99.00%
2. 最終多數決整合後，未知池的恶章意圖分佈如下：
最終集成標籤
借貸融資       177
黃色與特種行業     18
博弈           4
詐騙高風險招募      1
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
